##**RAG**
Retrival Augumentation Generation

**To Implement RAG, We need to Install Some dependencies**

In [ ]:
!pip install -qU langchain-community langchain-google-genai langchain chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0

In [ ]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.2 MB/s eta 0:00:00


**Import All Necessary Libraries**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader # It loads the .txt files
from langchain_text_splitters import RecursiveCharacterTextSplitter # It creates chunks of my data
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI # For Embedding Model, Chat Model
from langchain_community.vectorstores import Chroma # This will help us to create Vector Store
from langchain_core.prompts import ChatPromptTemplate # This will help us to create system prompt for LLM
from langchain_core.output_parsers import StrOutputParser #This will help us to parse the output of model in the string format
from langchain_core.runnables import RunnablePassthrough # this allow us to take input and pass it to our prompts

In [ ]:
import os

In [ ]:
GOOGLE_API_KEY = 'YOUR_GOOGLE_API_KEY'

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

**We will be creating Rag Module by Module**

**Step 1: Load and Splitting of Document**

In [ ]:
def load_and_split(filepath):
  loader = PyPDFLoader(filepath) # It is an object of TextLoader
  docs = loader.load() # It will load the content from the doc / lazyload() --> Very large Data ---> LazyLoad()

  print('Splitting Data into Chunks...')
  splitter = RecursiveCharacterTextSplitter(chunk_size = 2000, chunk_overlap = 500)
  splits = splitter.split_documents(docs)
  print(f'Splits {len(splits)} Chunks')
  return splits

**We Will Create Rag Chain**

In [ ]:
def create_rag_chain(splits):
  # We need to Embed the data into vectors
  # taskType = 'RETRIEVAL_DOCUMENT' for embedding the docs
  embeddings = GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-001',task_type='RETRIEVAL_DOCUMENT')
  # we will pass the embeddings to my vector store
  vectorStore = Chroma.from_documents(documents=splits, embedding =embeddings )
  # Now we have to create a retriver
  retriver = vectorStore.as_retriever() # this is going to find context from my Vector DataBase

  # Since our retriver is ready, Lets create our LLM
  llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature= 1.5)
  # temperature (0-2): It allows us to increase or decrease the creativeness of the model
  template = """Answer the question based only on the following Context: {context}
                Question: {question}

                Helpful Answer: """

  sysPrompt = ChatPromptTemplate.from_template(template) # System Prompt

  # We will Create a function that will join the retrived chunks into one doc then we will pass that doc as context
  def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

  # Rag Chain

  # Retriver --> Take QUestion --> Embed it ---> Retrive from Vector Store ---> Format_docs --> It will Passed as contet
            # +
  # RunnablePassThrough --> Take question ---> Question Placeholder ===> SysPrompt --> LLM ---> Output
  chain = (
      {'context': retriver |format_docs, 'question': RunnablePassthrough() }
      | sysPrompt
      | llm
      | StrOutputParser()
  )

  return chain

In [ ]:
filePath = '/content/RayOptics.pdf'

splits = load_and_split(filePath)

Splitting Data into Chunks...
Splits 52 Chunks


In [ ]:
rag_chain = create_rag_chain(splits)

In [ ]:
Question = input('Ask Question Related to Ray Optics: ')

output = rag_chain.invoke(input = Question)
print(output)

Ask Question Related to Ray Optics: What is the law of reflection?
The laws of reflection state that:
1.  The angle of reflection equals the angle of incidence.
2.  The incident ray, reflected ray, and the normal to the reflecting surface at the point of incidence lie in the same plane (Fig. 9.1).


In [ ]:
Question = input('Ask Question Related to Ray Optics: ')

output = rag_chain.invoke(input = Question)
print(output)

Ask Question Related to Ray Optics: Who is balreddy?
The provided text does not contain any information about "balreddy".


**Bonus**

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
def response(message, history):
  return rag_chain.invoke(message)

In [ ]:
demo = gr.ChatInterface(
    fn = response,
    textbox = gr.Textbox(placeholder='Ask a question regarding Ray Optics')
)

demo.launch(share = True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://210854713405106ce3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
